# Trading Intelligent — CH03 v2.1 — Notebook 2
## Asymétrie, prévision hors échantillon, VaR et Expected Shortfall

**Auteur académique : Tawfik Masrour**

### Question directrice
Un modèle plus riche améliore-t-il réellement la mesure du risque lorsque l'on respecte un **futur de test intact** ?

### Objectifs
1. séparer chronologiquement entraînement et test ;
2. comparer GARCH et GJR-GARCH ;
3. estimer l'épaisseur de queue d'une Student-t standardisée ;
4. produire des prévisions one-step-ahead hors échantillon ;
5. calculer VaR et ES ;
6. contrôler couverture et indépendance des dépassements ;
7. utiliser `true_sigma` uniquement comme oracle pédagogique d'évaluation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import norm, t as student_t, chi2
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

SEED = 20261001
DATASET_ID = "TI_CH03_VOL_RISK_V1"
DATASET_FILENAME = "TI_CH03_Vol_Risk_v1.csv"
np.random.seed(SEED)


def generate_dataset(seed=SEED, n=1700):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2020-01-02", periods=n)
    mu = 0.00018
    alpha, beta, gamma = 0.055, 0.87, 0.10
    target_var = 0.00010
    nu = 7
    omega = target_var * (1 - alpha - beta - gamma / 2)
    z = rng.standard_t(df=nu, size=n) / np.sqrt(nu / (nu - 2))
    eps = np.zeros(n)
    sig2 = np.full(n, target_var)
    r = np.zeros(n)
    for i in range(1, n):
        indicator = 1.0 if eps[i - 1] < 0 else 0.0
        sig2[i] = (
            omega
            + alpha * eps[i - 1] ** 2
            + gamma * indicator * eps[i - 1] ** 2
            + beta * sig2[i - 1]
        )
        eps[i] = np.sqrt(sig2[i]) * z[i]
        r[i] = mu + eps[i]
    price = 100 * np.exp(np.cumsum(r))
    volume = np.exp(
        15.7 + 18 * np.sqrt(sig2) + 45 * np.abs(r) + rng.normal(0, 0.25, n)
    ).astype("int64")
    return pd.DataFrame({
        "date": dates,
        "return": r,
        "price": price,
        "true_sigma": np.sqrt(sig2),
        "innovation": eps,
        "volume": volume,
    })


def locate_dataset(filename=DATASET_FILENAME):
    here = Path.cwd()
    roots = [here, here.parent, here.parent.parent]
    subdirs = [Path('.'), Path('datasets'), Path('data'), Path('resources')]
    for root in roots:
        for sub in subdirs:
            candidate = (root / sub / filename).resolve()
            if candidate.exists():
                return candidate
    return None


path = locate_dataset()
if path is not None:
    df = pd.read_csv(path, parse_dates=['date'])
    print(f"{DATASET_ID} chargé depuis : {path}")
else:
    df = generate_dataset()
    print(f"{DATASET_ID} absent localement : version déterministe régénérée en mémoire.")
print("Dimensions :", df.shape)
print("Convention : return et true_sigma sont stockés en décimal.")


def garch_variance(residuals, omega, alpha, beta):
    e = np.asarray(residuals, float)
    v = np.empty_like(e)
    v[0] = max(np.var(e), 1e-8)
    for i in range(1, len(e)):
        v[i] = omega + alpha * e[i - 1] ** 2 + beta * v[i - 1]
    return v


def fit_garch11(r_dec):
    x = np.asarray(r_dec, float) * 100.0  # estimation en pourcentage
    mu = x.mean()
    e = x - mu
    sample_var = max(np.var(e), 1e-6)

    def objective(p):
        omega, alpha, beta = p
        if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 0.999:
            return 1e12
        v = garch_variance(e, omega, alpha, beta)
        return 0.5 * np.sum(np.log(2 * np.pi) + np.log(v) + e ** 2 / v)

    result = minimize(
        objective,
        [0.03 * sample_var, 0.08, 0.88],
        method='SLSQP',
        bounds=[(1e-10, None), (0, 0.999), (0, 0.999)],
        constraints=[{'type': 'ineq', 'fun': lambda p: 0.999 - p[1] - p[2]}],
        options={'maxiter': 1200, 'ftol': 1e-10},
    )
    omega, alpha, beta = result.x
    v = garch_variance(e, omega, alpha, beta)
    return {
        'success': bool(result.success),
        'message': str(result.message),
        'mu': mu,
        'omega': omega,
        'alpha': alpha,
        'beta': beta,
        'variance': v,
        'nll': float(result.fun),
        'nobs': len(x),
    }


def gjr_variance(residuals, omega, alpha, gamma, beta):
    e = np.asarray(residuals, float)
    v = np.empty_like(e)
    v[0] = max(np.var(e), 1e-8)
    for i in range(1, len(e)):
        prev = e[i - 1]
        v[i] = (
            omega
            + alpha * prev ** 2
            + gamma * (prev < 0) * prev ** 2
            + beta * v[i - 1]
        )
    return v


def fit_gjr11(r_dec):
    x = np.asarray(r_dec, float) * 100.0
    mu = x.mean()
    e = x - mu
    sample_var = max(np.var(e), 1e-6)

    def objective(p):
        omega, alpha, gamma, beta = p
        if min(omega, alpha, gamma, beta) < 0:
            return 1e12
        if alpha + beta + gamma / 2 >= 0.999:
            return 1e12
        v = gjr_variance(e, omega, alpha, gamma, beta)
        return 0.5 * np.sum(np.log(2 * np.pi) + np.log(v) + e ** 2 / v)

    starts = [
        [0.025 * sample_var, 0.05, 0.10, 0.85],
        [0.030 * sample_var, 0.08, 0.08, 0.82],
        [0.020 * sample_var, 0.04, 0.12, 0.86],
    ]
    candidates = [
        minimize(
            objective,
            start,
            method='Nelder-Mead',
            options={'maxiter': 8000, 'xatol': 1e-10, 'fatol': 1e-8},
        )
        for start in starts
    ]
    result = min(candidates, key=lambda res: res.fun)
    omega, alpha, gamma, beta = result.x
    v = gjr_variance(e, omega, alpha, gamma, beta)
    return {
        'success': bool(result.success),
        'message': str(result.message),
        'mu': mu,
        'omega': omega,
        'alpha': alpha,
        'gamma': gamma,
        'beta': beta,
        'variance': v,
        'nll': float(result.fun),
        'nobs': len(x),
    }


def fit_standardized_t_nu(z):
    z = np.asarray(z, float)

    def nll(nu):
        if nu <= 2.05:
            return 1e12
        scale = np.sqrt(nu / (nu - 2))
        return -np.sum(np.log(scale) + student_t.logpdf(scale * z, df=nu))

    result = minimize_scalar(nll, bounds=(2.1, 80), method='bounded')
    return float(result.x)


def forecast_gjr_one_step(test_returns_dec, fit, last_train_return_dec):
    """Prévisions one-step-ahead avec paramètres figés et mise à jour par le passé observé."""
    test_x = np.asarray(test_returns_dec, float) * 100.0
    mu = fit['mu']
    omega, alpha, gamma, beta = fit['omega'], fit['alpha'], fit['gamma'], fit['beta']
    prev_v = float(fit['variance'][-1])
    prev_e = float(last_train_return_dec * 100.0 - mu)
    out = np.empty(len(test_x))
    for i, x in enumerate(test_x):
        out[i] = omega + alpha * prev_e ** 2 + gamma * (prev_e < 0) * prev_e ** 2 + beta * prev_v
        prev_e = x - mu
        prev_v = out[i]
    return out


def safe_xlogy(count, probability):
    if count == 0:
        return 0.0
    if probability <= 0:
        return -np.inf
    if probability >= 1:
        return 0.0 if np.isclose(probability, 1.0) else -np.inf
    return count * np.log(probability)


def kupiec_test(breaches, alpha_tail):
    hits = np.asarray(breaches, dtype=int)
    n = len(hits)
    x = int(hits.sum())
    p_hat = x / n
    ll0 = safe_xlogy(x, alpha_tail) + safe_xlogy(n - x, 1 - alpha_tail)
    if p_hat in (0, 1):
        ll1 = 0.0
    else:
        ll1 = safe_xlogy(x, p_hat) + safe_xlogy(n - x, 1 - p_hat)
    lr = -2 * (ll0 - ll1)
    return float(lr), float(1 - chi2.cdf(lr, 1))


def christoffersen_independence_test(breaches):
    h = np.asarray(breaches, dtype=int)
    if len(h) < 2:
        return np.nan, np.nan
    n00 = int(np.sum((h[:-1] == 0) & (h[1:] == 0)))
    n01 = int(np.sum((h[:-1] == 0) & (h[1:] == 1)))
    n10 = int(np.sum((h[:-1] == 1) & (h[1:] == 0)))
    n11 = int(np.sum((h[:-1] == 1) & (h[1:] == 1)))
    total = n00 + n01 + n10 + n11
    p = (n01 + n11) / total if total else 0.0
    p01 = n01 / (n00 + n01) if (n00 + n01) else 0.0
    p11 = n11 / (n10 + n11) if (n10 + n11) else 0.0
    ll0 = safe_xlogy(n01 + n11, p) + safe_xlogy(n00 + n10, 1 - p)
    ll1 = (
        safe_xlogy(n01, p01) + safe_xlogy(n00, 1 - p01)
        + safe_xlogy(n11, p11) + safe_xlogy(n10, 1 - p11)
    )
    if not np.isfinite(ll0) or not np.isfinite(ll1):
        return np.nan, np.nan
    lr = -2 * (ll0 - ll1)
    return float(lr), float(1 - chi2.cdf(lr, 1))


def qlike(realized_variance, forecast_variance):
    x = np.asarray(realized_variance, float)
    v = np.asarray(forecast_variance, float)
    ratio = np.maximum(x, 1e-12) / np.maximum(v, 1e-12)
    return ratio - np.log(ratio) - 1


## 1. Figer le futur avant de modéliser

Nous réservons les 25 % dernières observations comme **test chronologique**. Aucun paramètre n'est estimé sur cette partie.


In [ ]:
split = int(0.75 * len(df))
train = df.iloc[:split].copy()
test = df.iloc[split:].copy()
print("Train :", train.date.min().date(), "→", train.date.max().date(), len(train), "observations")
print("Test  :", test.date.min().date(), "→", test.date.max().date(), len(test), "observations")

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(train.date, 100 * train["return"], lw=0.55, label="Entraînement")
ax.plot(test.date, 100 * test["return"], lw=0.65, label="Test futur")
ax.axvline(test.date.iloc[0], color="black", ls="--", lw=1)
ax.set_ylabel("Rendement (%)")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


## 2. GARCH symétrique vs GJR-GARCH

Le GJR-GARCH ajoute un effet spécifique après un choc négatif. Pour cette convention, une condition de persistance courante sous innovations symétriques est
$$
\alpha+\beta+\frac{\gamma}{2}<1.
$$


In [ ]:
g = fit_garch11(train["return"])
j = fit_gjr11(train["return"])
comparison = pd.DataFrame({
    "GARCH(1,1)": [g["success"], g["nll"], g["omega"], g["alpha"], np.nan, g["beta"], 2*3 + 2*g["nll"]],
    "GJR-GARCH(1,1)": [j["success"], j["nll"], j["omega"], j["alpha"], j["gamma"], j["beta"], 2*4 + 2*j["nll"]],
}, index=["Convergence", "NLL", "omega", "alpha", "gamma", "beta", "AIC variance"])
comparison


## 3. Que reste-t-il dans les innovations standardisées ?

La dynamique de variance et la forme de la distribution sont deux problèmes différents. Nous estimons donc $\nu$ sur les résidus standardisés du **train uniquement**.


In [ ]:
train_e = 100 * train["return"].to_numpy() - j["mu"]
train_sigma = np.sqrt(j["variance"])
z_train = train_e / train_sigma
nu_hat = fit_standardized_t_nu(z_train)

print(pd.Series(z_train).describe(percentiles=[0.01, 0.05, 0.95, 0.99]))
print("Excès de kurtosis empirique :", pd.Series(z_train).kurtosis())
print("nu Student-t estimé         :", round(nu_hat, 3))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(z_train, bins=60, density=True, alpha=0.65, label="Résidus standardisés — train")
xx = np.linspace(-5, 5, 500)
ax.plot(xx, norm.pdf(xx), label="Normale")
ax.legend(); ax.grid(alpha=0.2); plt.show()


## 4. Prévision one-step-ahead sur le test

Les paramètres restent figés après l'entraînement. À chaque date de test, le rendement déjà observé peut mettre à jour la variance du lendemain : c'est une prévision one-step-ahead récursive, sans fuite du futur.


In [ ]:
test_v_gjr = forecast_gjr_one_step(
    test["return"].to_numpy(),
    j,
    train["return"].iloc[-1],
)
test_sigma_gjr_pct = np.sqrt(test_v_gjr)
test_sigma_gjr_dec = test_sigma_gjr_pct / 100

# Oracle pédagogique : true_sigma n'intervient qu'après la prévision.
mae_sigma = np.mean(np.abs(test_sigma_gjr_dec - test["true_sigma"].to_numpy()))
qlike_oracle = np.mean(qlike(test["true_sigma"].to_numpy() ** 2, (test_sigma_gjr_dec ** 2)))
print(f"MAE sigma vs oracle : {mae_sigma:.6f}")
print(f"QLIKE vs oracle     : {qlike_oracle:.6f}")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(test.date, 100 * test.true_sigma, label="Volatilité vraie — oracle", lw=1.3)
ax.plot(test.date, test_sigma_gjr_pct, label="Prévision GJR one-step-ahead", lw=1.0)
ax.set_ylabel("Volatilité (%)"); ax.legend(); ax.grid(alpha=0.2); plt.show()


## 5. Prévision GARCH multi-horizon

À partir d'un état donné, la variance GARCH stationnaire converge vers sa variance de long terme. Cette courbe décrit une **trajectoire conditionnelle prévue**, pas une certitude.


In [ ]:
p = g["alpha"] + g["beta"]
long_var = g["omega"] / (1 - p)
last_v = g["variance"][-1]
last_e = 100 * train["return"].iloc[-1] - g["mu"]
v1 = g["omega"] + g["alpha"] * last_e ** 2 + g["beta"] * last_v
h = np.arange(1, 61)
vh = long_var + (p ** (h - 1)) * (v1 - long_var)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(h, np.sqrt(vh))
ax.axhline(np.sqrt(long_var), ls="--", label="Long terme")
ax.set(xlabel="Horizon (jours)", ylabel="Volatilité prévue (%)", title="Prévision multi-horizon")
ax.legend(); ax.grid(alpha=0.2); plt.show()


## 6. VaR et Expected Shortfall sur le futur de test

On définit la perte $L_t=-r_t$. Pour un niveau de confiance de 99 %, la probabilité de queue vaut $\alpha=1\%$.

Nous comparons deux distributions **avec la même volatilité prévue** : normale et Student-t standardisée.


In [ ]:
alpha_tail = 0.01
mu_dec = j["mu"] / 100
sigma_dec = test_sigma_gjr_dec

# Normal
qN = norm.ppf(alpha_tail)
var99_normal = -(mu_dec + sigma_dec * qN)
es_left_N = -norm.pdf(qN) / alpha_tail
es99_normal = -(mu_dec + sigma_dec * es_left_N)

# Student-t standardisée
scale_t = np.sqrt(nu_hat / (nu_hat - 2))
qt_raw = student_t.ppf(alpha_tail, df=nu_hat)
qt = qt_raw / scale_t
es_left_T = -(
    student_t.pdf(qt_raw, df=nu_hat)
    * (nu_hat + qt_raw ** 2)
    / ((nu_hat - 1) * alpha_tail)
) / scale_t
var99_t = -(mu_dec + sigma_dec * qt)
es99_t = -(mu_dec + sigma_dec * es_left_T)

print("Dernière VaR 99 % normale   :", round(100 * var99_normal[-1], 3), "%")
print("Dernier ES 99 % normal      :", round(100 * es99_normal[-1], 3), "%")
print("Dernière VaR 99 % Student-t :", round(100 * var99_t[-1], 3), "%")
print("Dernier ES 99 % Student-t   :", round(100 * es99_t[-1], 3), "%")


## 7. Backtest : fréquence **et** indépendance

Un taux proche de 1 % est nécessaire pour une VaR 99 %, mais pas suffisant. Des exceptions regroupées peuvent révéler un modèle qui réagit trop lentement.


In [ ]:
loss_test = -test["return"].to_numpy()
bN = loss_test > var99_normal
bT = loss_test > var99_t

rows = []
for name, breaches in [("Normal", bN), ("Student-t", bT)]:
    lr_uc, p_uc = kupiec_test(breaches, alpha_tail)
    lr_ind, p_ind = christoffersen_independence_test(breaches)
    rows.append({
        "Modèle": name,
        "Dépassements": int(breaches.sum()),
        "Taux (%)": 100 * breaches.mean(),
        "Kupiec p-value": p_uc,
        "Indépendance p-value": p_ind,
    })
results = pd.DataFrame(rows).set_index("Modèle")
results


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(test.date, 100 * loss_test, lw=0.6, label="Perte observée")
ax.plot(test.date, 100 * var99_t, lw=1.1, label="VaR 99 % Student-t")
ax.scatter(test.date[bT], 100 * loss_test[bT], s=24, label="Dépassement", zorder=3)
ax.set_ylabel("Perte (%)")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


## 8. Mini-défi étudiant

1. Comparez Normal et Student-t sur le **test seulement**. Lequel paraît mieux calibré ici et pourquoi ?
2. Remplacez 99 % par 95 %. Comment évoluent VaR, ES et nombre de dépassements ?
3. Expliquez pourquoi l'ES ne se backteste pas par un simple taux de dépassement.
4. Comparez la prévision GJR à `true_sigma`. Pourquoi cet oracle ne serait-il pas disponible dans un marché réel ?
5. Utilisez $r_t^2$ comme proxy de variance et comparez son bruit visuel à `true_sigma`.


In [ ]:
# Votre travail commence ici.
# Q1

# Q2

# Q3

# Q4

# Q5


### À retenir

Prévoir la volatilité n'est pas encore gérer le risque. Il faut ensuite choisir une distribution conditionnelle, une convention de perte, un horizon et un protocole de validation. La règle méthodologique la plus importante de ce notebook est simple : **estimer sur le passé, figer le modèle, puis évaluer sur un futur resté intact**.
